In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import numpy as np
import pandas as pd
import xarray as xr
import pingouin as pg
import plotly.graph_objects as go
from os.path import join as pjoin
from natsort import natsorted

sys.path.append("../../")
import circletrack_behavior as ctb
import plotting_functions as pf

In [ ]:
## Settings
parent_dir = 'CircleTrack_Inhibition'
experiment_dir = 'Inhibition2'
lin_path = f'../../../{parent_dir}/{experiment_dir}/output/lin_behav/'
circle_path = f'../../../{parent_dir}/{experiment_dir}/output/behav/'
fig_path = f'../../../{parent_dir}/{experiment_dir}/intermediate_figures'
maze_info = pd.read_csv(f'../../../{parent_dir}/{experiment_dir}/maze_yml/maze_info.csv')
chance_color = '#7d7d7d'
avg_color = 'midnightblue'
subject_color = 'darkgrey'
two_group_colors = ['darkorchid', 'midnightblue']
group_colors_dict = {'CNO': 'darkorchid', 'Saline': 'midnightblue'}
error_dict = {'CNO': 'rgba(153,50,204,0.4)', 'Saline': 'rgba(0,41,102,0.4)'}

if not os.path.exists(fig_path):
    os.makedirs(fig_path)

### Circle track lick accuracy and rewards.

In [ ]:
circletrack_results = {'mouse': [], 'day': [], 'sex': [], 'group': [], 'first_recall_group': [], 'session': [], 'lick_accuracy': [], 'rewards': []}
for mouse in os.listdir(circle_path):
    mouse_path = pjoin(circle_path, mouse)
    sex = maze_info['Sex'][maze_info['Mouse'] == mouse].values[0]
    group = maze_info['Group'][maze_info['Mouse'] == mouse].values[0]
    first_recall = maze_info['Recall1_Group'][maze_info['Mouse'] == mouse].values[0]
    for idx, session in enumerate(os.listdir(mouse_path)):
        behav = pd.read_feather(pjoin(mouse_path, f'{session}'))
        behav = behav[~behav['probe']]
        reward_one, reward_two = np.unique(behav['reward_one'])[0], np.unique(behav['reward_two'])[0]
        pc_thresh5 = ctb.lick_accuracy(behav, port_list=[reward_one, reward_two], lick_threshold=5, by_trials=False)
        circletrack_results['mouse'].append(mouse)
        circletrack_results['day'].append(idx+1)
        circletrack_results['sex'].append(sex)
        circletrack_results['group'].append(group)
        circletrack_results['first_recall_group'].append(first_recall)
        circletrack_results['session'].append(np.unique(behav['session'])[0])
        circletrack_results['lick_accuracy'].append(pc_thresh5)
        circletrack_results['rewards'].append(np.sum(behav['water']))
ct_df = pd.DataFrame(circletrack_results)

In [ ]:
## Plot 5th lick accuracy across days
fig = pf.plot_behavior_across_days(ct_df, x_var='day', y_var='lick_accuracy', groupby_var=['day'], plot_transitions=None,
                                   marker_color=avg_color, avg_color=avg_color, expert_line=False, chance=True, symbols=['circle'],
                                   x_title='Day', y_title='Lick Accuracy (%)', titles=['Circle Track'], height=500, width=500)
fig.update_yaxes(range=[0, 100])
fig.show()

In [ ]:
## Plot 5th lick accuracy across days for both groups up to day 4 in B
fig = pf.plot_behavior_across_days(ct_df[ct_df['day'] < 10], x_var='day', y_var='lick_accuracy', groupby_var=['day', 'group'], plot_transitions=[5.5], transition_color=['darkgrey'],
                                   marker_color=two_group_colors, avg_color=avg_color, expert_line=False, chance=True, symbols=['circle', 'circle'], plot_datapoints=False,
                                   x_title='Day', y_title='Lick Accuracy (%)', titles=['Circle Track'], height=500, width=500)
fig.update_yaxes(range=[0, 100])
fig.show()
fig.write_image(pjoin(fig_path, 'lick_accuracy_across_days.png'), width=500, height=500)

In [ ]:
## Plot 5th lick accuracy across days for both groups up to day 6
fig = pf.plot_behavior_across_days(ct_df[ct_df['day'] <= 6], x_var='day', y_var='lick_accuracy', groupby_var=['day', 'group'], plot_transitions=[5.5], transition_color=['darkgrey'],
                                   marker_color=two_group_colors, avg_color=avg_color, expert_line=False, chance=True, symbols=['circle', 'circle'], plot_datapoints=False,
                                   x_title='Day', y_title='Lick Accuracy (%)', titles=[''], height=500, width=500)
fig.update_yaxes(range=[0, 100])
fig.add_vrect(x0=5.5, x1=6.5, fillcolor=chance_color, layer='below', opacity=0.5, line_width=0)
fig.update_layout(legend_traceorder='reversed')
fig.show()
fig.write_image(pjoin(fig_path, 'lick_acc_up_to_day6.png'), width=500, height=500)

## T-test of difference in means on day 6
pg.ttest(x=ct_df['lick_accuracy'][(ct_df['day'] == 6) & (ct_df['group'] == 'CNO')], y=ct_df['lick_accuracy'][(ct_df['day'] == 6) & (ct_df['group'] == 'Saline')])

In [ ]:
## Plot rewards across days
fig = pf.plot_behavior_across_days(ct_df, x_var='day', y_var='rewards', groupby_var=['day'], plot_transitions=None,
                                   marker_color=avg_color, avg_color=avg_color, expert_line=False, chance=False, symbols=['circle'],
                                   x_title='Day', y_title='Rewards', titles=['Circle Track'], height=500, width=500)
fig.show()

In [ ]:
## Plot rewards across days for both groups
fig = pf.plot_behavior_across_days(ct_df, x_var='day', y_var='rewards', groupby_var=['day', 'group'], plot_transitions=[5.5], transition_color=['darkgrey'],
                                   marker_color=two_group_colors, avg_color=avg_color, expert_line=False, chance=False, symbols=['circle', 'circle'], plot_datapoints=False,
                                   x_title='Day', y_title='Rewards', titles=['Circle Track'], height=500, width=500)
fig.show()
fig.write_image(pjoin(fig_path, 'rewards_across_days.png'), width=500, height=500)

In [ ]:
## Plot rewards across days for both groups up to day 6
fig = pf.plot_behavior_across_days(ct_df[ct_df['day'] <= 6], x_var='day', y_var='rewards', groupby_var=['day', 'group'], plot_transitions=[5.5], transition_color=['darkgrey'],
                                   marker_color=two_group_colors, avg_color=avg_color, expert_line=False, chance=False, symbols=['circle', 'circle'], plot_datapoints=False,
                                   x_title='Day', y_title='Rewards', titles=[''], height=500, width=500)
fig.update_yaxes(range=[0, 100])
fig.add_vrect(x0=5.5, x1=6.5, fillcolor=chance_color, layer='below', opacity=0.5, line_width=0)
fig.update_layout(legend_traceorder='reversed')
fig.show()
fig.write_image(pjoin(fig_path, 'rewards_up_to_day6.png'), width=500, height=500)

### Look at probe accuracy.

In [ ]:
lick_dict_probe = {'mouse': [], 'experiment': [], 'sex': [], 'group': [], 'first_recall_group': [], 'second_recall_group': [], 'session': [], 
                   'day': [], 'num_licks': [], 'probe_acc': [], 'session_acc': [], 'rewards': []}
for mouse in os.listdir(circle_path):
    mpath = pjoin(circle_path, mouse)
    sex = maze_info['Sex'][maze_info['Mouse'] == mouse].values[0]
    group = maze_info['Group'][maze_info['Mouse'] == mouse].values[0]
    first_recall = maze_info['Recall1_Group'][maze_info['Mouse'] == mouse].values[0]
    second_recall = maze_info['Recall2_Group'][maze_info['Mouse'] == mouse].values[0]
    for idx, session in enumerate(os.listdir(mpath)):
        behav = pd.read_feather(pjoin(mpath, session))
        if any(behav['probe']):
            behav_probe = behav[behav['probe']]
            behav_no_probe = behav[~behav['probe']]
            reward_one, reward_two = np.unique(behav['reward_one'])[0], np.unique(behav['reward_two'])[0]
            percent_correct = ctb.lick_accuracy(behav_probe, port_list=[reward_one, reward_two], lick_threshold=5, by_trials=False)
            session_pc = ctb.lick_accuracy(behav_no_probe, port_list=[reward_one, reward_two], lick_threshold=5, by_trials=False)
            lick_dict_probe['mouse'].append(mouse)
            lick_dict_probe['experiment'].append(behav['cohort'].unique()[0])
            lick_dict_probe['sex'].append(sex)
            lick_dict_probe['group'].append(group)
            lick_dict_probe['first_recall_group'].append(first_recall)
            lick_dict_probe['second_recall_group'].append(second_recall)
            lick_dict_probe['day'].append(idx+1)
            lick_dict_probe['session'].append(np.unique(behav['session_two'])[0])
            lick_dict_probe['num_licks'].append(len(behav_probe[behav_probe['lick_port'] != -1]))
            lick_dict_probe['probe_acc'].append(percent_correct)
            lick_dict_probe['session_acc'].append(session_pc)
            lick_dict_probe['rewards'].append(np.sum(behav_no_probe['water']))
        else:
            pass
probe_df = pd.DataFrame(lick_dict_probe)

In [ ]:
## Plot probe accuracy for Recall1 and Recall2 separated by if mice had gotten CNO during first day in B
sub_lines = {'CNO:Saline': 'dash', 'Saline:CNO': 'dot'}
mf_colors = {'M': 'midnightblue', 'F': 'orchid'}
fig = pf.custom_graph_template(x_title='Day', y_title='', titles=['Saline on B1', 'CNO on B1'], 
                               rows=1, columns=2, shared_x=True, shared_y=True, width=800)

for mouse in probe_df['mouse'].unique():
    mdata = probe_df[probe_df['mouse'] == mouse]
    mdata = mdata[(mdata['day'] == 10) | (mdata['day'] == 12)]

    if mdata['group'].unique()[0] == 'Saline':
        col = 1
    else:
        col = 2
    
    recall_group = f'{mdata['first_recall_group'].unique()[0]}:{mdata['second_recall_group'].unique()[0]}'
    
    fig.add_trace(go.Scattergl(x=mdata['day'].astype(str), y=mdata['probe_acc'], mode='lines', line=dict(dash=sub_lines[recall_group]),
                               line_color=mf_colors[mdata['sex'].unique()[0]], name=recall_group, legendgroup=recall_group, showlegend=False), row=1, col=col)
fig.add_hline(y=25, line_width=3, line_dash='dash', line_color=chance_color, opacity=1)
fig.update_yaxes(range=[0, 101])
fig.update_yaxes(title='Lick Accuracy (%)', col=1)
fig.data[0]['showlegend'] = True
fig.data[1]['showlegend'] = True
fig.show()

### Look at probe accuracy across trials.

In [ ]:
bin_size = 1
lick_dict_trials = {'mouse': [], 'sex': [], 'group': [], 'first_recall_group': [], 'session': [], 
                   'day': [], 'trial': [], 'lick_acc': []}
for mouse in os.listdir(circle_path):
    mpath = pjoin(circle_path, mouse)
    sex = maze_info['Sex'][maze_info['Mouse'] == mouse].values[0]
    group = maze_info['Group'][maze_info['Mouse'] == mouse].values[0]
    first_recall = maze_info['Recall1_Group'][maze_info['Mouse'] == mouse].values[0]
    for idx, session in enumerate(os.listdir(mpath)):
        behav = pd.read_feather(pjoin(mpath, session))
        if any(behav['probe']):
            behav_probe = behav[behav['probe']]
            reward_one, reward_two = behav_probe['reward_one'].unique()[0], behav_probe['reward_two'].unique()[0]
            trial_acc = ctb.lick_accuracy(behav_probe, port_list=[reward_one, reward_two], lick_threshold=5, by_trials=True)

            if bin_size > 1:
                binned_acc = ctb.bin_data(trial_acc, bin_size)
            else:
                binned_acc = trial_acc
            
            for trial, val in enumerate(binned_acc):
                lick_dict_trials['mouse'].append(mouse)
                lick_dict_trials['sex'].append(sex)
                lick_dict_trials['group'].append(group)
                lick_dict_trials['first_recall_group'].append(first_recall)
                lick_dict_trials['day'].append(idx+1)
                lick_dict_trials['session'].append(behav_probe['session_two'].unique()[0])
                lick_dict_trials['trial'].append(trial+1)
                lick_dict_trials['lick_acc'].append(val)
probe_trials_df = pd.DataFrame(lick_dict_trials)

In [ ]:
sub = probe_trials_df[probe_trials_df['day'] == 10]
sub.groupby(['group', 'first_recall_group', 'trial'], as_index=False).agg({'lick_acc': ['mean', 'sem']})

### Look at lick accuracy across trials.

In [ ]:
bin_size = 3
lick_thresh = 5
trial_res = {'mouse': [], 'sex': [], 'group': [], 'day': [], 'session_two': [], 'trial': [], 'lick_acc': []}
for mouse in os.listdir(circle_path):
    mpath = pjoin(circle_path, mouse)
    sex = maze_info['Sex'][maze_info['Mouse'] == mouse].values[0]
    group = maze_info['Group'][maze_info['Mouse'] == mouse].values[0]
    for idx, session in enumerate(natsorted(os.listdir(mpath))):
        behav = pd.read_feather(pjoin(mpath, f'{session}'))
        reward_one, reward_two = behav['reward_one'].unique()[0], behav['reward_two'].unique()[0]
        trial_acc = ctb.lick_accuracy(behav, port_list=[reward_one, reward_two], lick_threshold=lick_thresh, by_trials=True)
        
        if bin_size > 1:
            binned_acc = ctb.bin_data(trial_acc, bin_size)
        else:
            binned_acc = trial_acc

        for trial, val in enumerate(binned_acc):
            trial_res['mouse'].append(mouse)
            trial_res['sex'].append(sex)
            trial_res['group'].append(group)
            trial_res['day'].append(idx+1)
            trial_res['session_two'].append(behav['session_two'].unique()[0])
            trial_res['trial'].append(trial+1)
            trial_res['lick_acc'].append(val)
trial_df = pd.DataFrame(trial_res)

In [ ]:
## Plot accuracy across first three days in A
only_min_trials = False
ar = trial_df.copy()
avg_ar = ar.groupby(['group', 'session_two', 'trial'], as_index=False).agg({'lick_acc': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='', y_title='', rows=1, columns=5, shared_x=True, width=1000, height=500,
                               shared_y=True, titles=[f'A{x}' for x in np.arange(1, 6)])
for group in ['Saline', 'CNO']:
    gdata = avg_ar[avg_ar['group'] == group].reset_index(drop=True)  
    for sess in ['A1', 'A2', 'A3', 'A4', 'A5']:
        trial_bins = []
        if only_min_trials:
            int_data = ar[(ar['group'] == group) & (ar['session_two'] == sess)]
            for mouse in int_data['mouse'].unique():
                d = int_data[int_data['mouse'] == mouse].reset_index(drop=True)
                trial_bins.append(d['trial'].shape[0])
                min_trials = np.min(trial_bins)
        else:
            min_trials = None

        if min_trials is not None:
            sess_data = gdata[gdata['session_two'] == sess].reset_index(drop=True)[:min_trials]
            xaxis = np.arange(1, sess_data['trial'].to_numpy()[-1] * bin_size, bin_size)[:min_trials]
            upper = sess_data['lick_acc']['mean'] + sess_data['lick_acc']['sem']
            lower = sess_data['lick_acc']['mean'] - sess_data['lick_acc']['sem']
        else:
            sess_data = gdata[gdata['session_two'] == sess].reset_index(drop=True)
            xaxis = np.arange(1, sess_data['trial'].to_numpy()[-1] * bin_size, bin_size)
            upper = sess_data['lick_acc']['mean'] + sess_data['lick_acc']['sem']
            lower = sess_data['lick_acc']['mean'] - sess_data['lick_acc']['sem']
        fig.add_trace(go.Scattergl(x=xaxis, y=sess_data['lick_acc']['mean'], mode='lines', line_color=group_colors_dict[group],
                                   name=group, showlegend=False, legendgroup=group), row=1, col=int(sess[-1]))
        fig.add_trace(go.Scatter(x=xaxis, y=upper, mode='lines', marker=dict(color=error_dict[group]),
                                        name='Upper Bound', line=dict(width=0), showlegend=False), row=1, col=int(sess[-1]))
        fig.add_trace(go.Scatter(x=xaxis, y=lower, mode='lines', marker=dict(color=error_dict[group]),
                                name='Lower Bound', line=dict(width=0), showlegend=False, fillcolor=error_dict[group], fill='tonexty'), row=1, col=int(sess[-1]))
fig.add_hline(y=25, line_width=2, line_dash='dash', line_color='darkgrey', opacity=1)
fig.update_yaxes(range=[0, 100])
fig.update_yaxes(title='Lick Accuracy (%)', col=1)
fig.update_xaxes(title='Trials', row=1)
for val in [0, 15]:
    fig.data[val]['showlegend'] = True
fig.show()
fig.write_image(pjoin(fig_path, 'lick_accuracy_across_trials_A1_A5.png'), width=500, height=500)

In [ ]:
## Plot lick accuracy across trials on day 6
fig = pf.custom_graph_template(x_title='Trials', y_title='Lick Accuracy (%)', titles=['Day 6'], width=500)

avg = trial_df.groupby(['group', 'day', 'session_two', 'trial'], as_index=False).agg({'lick_acc': ['mean', 'sem']})
for group in ['Saline', 'CNO']:
    gdata = avg[(avg['group'] == group) & (avg['day'] == 6)]
    xaxis = np.arange(1, gdata['trial'].to_numpy()[-1] * bin_size, bin_size)
    upper = gdata['lick_acc']['mean'] + gdata['lick_acc']['sem']
    lower = gdata['lick_acc']['mean'] - gdata['lick_acc']['sem']

    fig.add_trace(go.Scattergl(x=xaxis, y=gdata['lick_acc']['mean'], mode='lines', line_color=group_colors_dict[group],
                                   name=group, showlegend=True, legendgroup=group))
    fig.add_trace(go.Scatter(x=xaxis, y=upper, mode='lines', marker=dict(color=error_dict[group]),
                                    name='Upper Bound', line=dict(width=0), showlegend=False))
    fig.add_trace(go.Scatter(x=xaxis, y=lower, mode='lines', marker=dict(color=error_dict[group]),
                            name='Lower Bound', line=dict(width=0), showlegend=False, fillcolor=error_dict[group], fill='tonexty'))
fig.add_hline(y=25, line_width=3, line_dash='dash', line_color='darkgrey', opacity=1, layer='below')
fig.show()
fig.write_image(pjoin(fig_path, 'accuracy_across_trials_B1.png'), width=500, height=500)

In [ ]:
anova_data = trial_df[(trial_df['day'] == 6) & (trial_df['trial'] <= 20)]
anova_data.loc[np.isnan(anova_data['lick_acc']), 'lick_acc'] = 0 ## set NaN to zero
anova_data.mixed_anova(dv='lick_acc', within='trial', between='group', subject='mouse')

In [ ]:
## Plot individual trial accuracy curves on day 6
fig = pf.custom_graph_template(x_title='Trials', y_title='Lick Accuracy (%)', titles=['B1'], width=600)

for mouse in trial_df['mouse'].unique():
    mdata = trial_df[(trial_df['mouse'] == mouse) & (trial_df['day'] == 6)]
    xaxis = np.arange(1, mdata['trial'].to_numpy()[-1] * bin_size, bin_size)
    fig.add_trace(go.Scattergl(x=xaxis, y=mdata['lick_acc'], mode='lines', line_color=group_colors_dict[mdata['group'].unique()[0]],
                               name=mouse, showlegend=True, line_width=3))
fig.add_hline(y=25, line_width=2, line_dash='dash', line_color='darkgrey', opacity=1)
fig.update_yaxes(range=[0, 100])
fig.show()

In [ ]:
## Plot number of trials ran
num_trials_dict = {'mouse': [], 'sex': [], 'group': [], 'day': [], 'num_trials': []}
for mouse in trial_df['mouse'].unique():
    for day in trial_df['day'].unique():
        mdata = trial_df[(trial_df['mouse'] == mouse) & (trial_df['day'] == day)]

        num_trials_dict['mouse'].append(mouse)
        num_trials_dict['sex'].append(mdata['sex'].unique()[0])
        num_trials_dict['group'].append(mdata['group'].unique()[0])
        num_trials_dict['day'].append(mdata['day'].unique()[0])
        num_trials_dict['num_trials'].append(mdata['trial'].to_numpy()[-1] * bin_size)
num_trials_df = pd.DataFrame(num_trials_dict)

In [ ]:
## Plot number of trials on day 6
day_of_int = 6
avg_trials = num_trials_df.groupby(['group', 'day'], as_index=False).agg({'num_trials': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='', y_title='Number of Trials')
for group in ['Saline', 'CNO']:
    gdata = avg_trials[(avg_trials['day'] == day_of_int) & (avg_trials['group'] == group)]
    fig.add_trace(go.Bar(x=gdata['group'], y=gdata['num_trials']['mean'], marker_line_width=2, marker_line_color='black', showlegend=False,
                         marker_color=group_colors_dict[group], error_y=dict(type='data', array=gdata['num_trials']['sem']), width=0.6))
fig.show()
fig.write_image(pjoin(fig_path, 'num_trials_day_6.png'), width=500, height=500)